In [5]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from typing import Any

from project_root import PROJECT_ROOT,DATASETS_ROOT

import fiftyone as fo
import fiftyone.utils.torch as fout


import torchvision as tv

from scripts.model_serialization import load_model

no_grad_guard = torch.no_grad()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
dataset_name =         "zoo-elephants-identity-tracks"
ds =fo.load_dataset(dataset_name)

classes = sorted(ds.classes["ground_truth"])
print(classes)
print(len(ds))

['01_Chandra', '02_Indi', '05_Thai']
4730


In [7]:
# Compute uniqueness
import fiftyone.brain as fob
dup_result = fob.compute_near_duplicates(ds)
print(len(dup_result.duplicate_ids))


session = fo.launch_app(ds, auto=False)
session.view = dup_result.duplicates_view()
session.open_tab()

Computing embeddings...
 100% |███████████████| 4730/4730 [13.1s elapsed, 0s remaining, 330.7 samples/s]      
Computing duplicate samples...
Generating index for 4730 embeddings...
Index complete
Generating neighbors graph for 4626 embeddings...
Index complete
Duplicates computation complete
104
Session launched. Run `session.show()` to open the App in a cell output.


<IPython.core.display.Javascript object>

In [ ]:
from tqdm.auto import tqdm
DELETE_DUPLICATES = False
if DELETE_DUPLICATES:
  for id in tqdm(dup_result.duplicate_ids):
    filepath = ds[id]["filepath"]
    Path.unlink(filepath)

  0%|          | 0/494 [00:00<?, ?it/s]

In [8]:
fob.compute_uniqueness(ds)
session.view = ds.sort_by("uniqueness")

 100% |████|  100.6Mb/100.6Mb [306.6ms elapsed, 0s remaining, 328.0Mb/s]      
Computing embeddings...
 100% |███████████████| 4730/4730 [1.6s elapsed, 0s remaining, 2.9K samples/s]        
Computing uniqueness...
Generating index for 4730 embeddings...
Index complete
Uniqueness computation complete


In [11]:
from fiftyone import ViewField as F
THRESHOLD = 0.01
ds_bad = ds.match(F("uniqueness") < 0.01)
print(f'Samples to drop: {len(ds_bad)}')
session.view = ds_bad

from tqdm.auto import tqdm
DELETE_BAD = True
if DELETE_BAD:
  for id in tqdm(dup_result.duplicate_ids):
    filepath = ds[id]["filepath"]
    Path.unlink(filepath)

Samples to drop: 253


  0%|          | 0/104 [00:00<?, ?it/s]